# SigLLM on Amazon-Book — full flow (data → MF → Q-Former stages → eval → ablation)

Focus metric: **uAUC** (per-user AUC). Baseline: **SeLLa-Rec** (arXiv:2504.10107),
which reuses the **CoLLM** Amazon-Book split — so we convert that split into this
repo's `*_ood2.pkl` schema and run the existing pipeline on it.

**Order:** run Section A once (data), then B (MF / CF substrate), then C–E (Q-Former
stages), then F (eval) and G (ablations). Training cells shell out to the existing
`sigllm.pipelines.*` entrypoints with `configs/config_amazon.yaml`.

## 0. Setup

Assumes the repo is at `REPO` and the conda/venv env is already installed
(mirror the env steps from `notebooks/SigLLM.ipynb` if needed).

In [ ]:
import os, sys, subprocess
REPO = '/content/SigLLM'            # repo root
SRC  = os.path.join(REPO, 'src')
CFG  = os.path.join(REPO, 'configs/config_amazon.yaml')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

def run(cmd):
    '''Run a shell command from REPO, streaming output.'''
    print('>>', cmd)
    return subprocess.run(cmd, shell=True, cwd=REPO, check=True)

## A. Amazon-Book data prep (reuse CoLLM / SeLLa-Rec split)

1. Point the paths below at your downloaded CoLLM Amazon-Book files.
2. **Inspect** them to see the real columns.
3. Fill the `ColumnMap` to match those columns.
4. **Convert** to `*_ood2.pkl`, then tag warm/cold and build the Q-Former pkls.

> CoLLM data: https://github.com/zyang1580/CoLLM (see the `collm-datasets` link).

In [ ]:
# A.1 — paths to the CoLLM/SeLLa-Rec Amazon-Book files you downloaded.
COLLM_DIR = os.path.join(REPO, 'data/raw/amazon-book')   # <-- put the files here
OUT_DIR   = os.path.join(REPO, 'data/processed/amazon-book')
os.makedirs(OUT_DIR, exist_ok=True)

PATHS = {
    'train':     os.path.join(COLLM_DIR, 'train.pkl'),     # adjust names as needed
    'valid':     os.path.join(COLLM_DIR, 'valid.pkl'),
    'test':      os.path.join(COLLM_DIR, 'test.pkl'),
    'item_text': os.path.join(COLLM_DIR, 'item_text.csv'), # iid -> title (+categories); or None
}

In [ ]:
# A.2 — inspect the real schema BEFORE converting.
from sigllm.datasets import inspect_collm_files
frames = inspect_collm_files(PATHS)

In [ ]:
# A.3 — map source columns -> canonical names (edit to match the inspect output).
from sigllm.datasets import ColumnMap

# Interaction files. Set `rating` instead of `label` if only a 1-5 score exists;
# set `his`/`his_title` only if the files already carry a history list.
cmap = ColumnMap(
    uid='uid', iid='iid', label='label', rating=None,
    timestamp='timestamp', his=None, his_title=None,
    title=None, genres=None,
)

# Item-text file (only `iid`, `title`, `genres` fields are used here).
item_text_map = ColumnMap(iid='iid', title='title', genres='categories')

In [ ]:
# A.4 — convert to this repo's *_ood2.pkl schema.
from sigllm.datasets import build_amazon_book
train_, valid_, test_, users_map, items_map = build_amazon_book(
    train_path=PATHS['train'], valid_path=PATHS['valid'], test_path=PATHS['test'],
    out_dir=OUT_DIR,
    item_text_path=PATHS.get('item_text'),
    cmap=cmap, item_text_map=item_text_map,
    rating_threshold=4.0,
    reuse_ids=False,   # set True if the CoLLM ids are already contiguous (0=pad)
)

In [ ]:
# A.5 — warm/cold tagging (unchanged from ML-1M pipeline).
from sigllm.datasets import process_warm_cold
_ = process_warm_cold(data_dir=OUT_DIR + '/', min_user_inter=3, min_item_inter=3)

In [ ]:
# A.6 — derive user_num / item_num and persist for later cells.
import pandas as pd
def _counts(out_dir):
    fr = [pd.read_pickle(os.path.join(out_dir, f'{s}_ood2.pkl')) for s in ('train','valid','test')]
    u = max(int(f['uid'].max()) for f in fr) + 1
    i = max(int(f['iid'].max()) for f in fr) + 1
    return u, i
USER_NUM, ITEM_NUM = _counts(OUT_DIR)
print('USER_NUM =', USER_NUM, ' ITEM_NUM =', ITEM_NUM)
OVR = f'model.rec_config.user_num={USER_NUM} model.rec_config.item_num={ITEM_NUM}'
print('override string:', OVR)

In [ ]:
# A.7 — build Q-Former alignment pkls (uses item_noun='book', rich_item_text=True
#        from config_amazon.yaml -> CHANGE 2d).
run(f'python -m sigllm.pipelines.multimodal.build_qformer_dataset --cfg-path {CFG}')

## B. MF baseline (CF substrate)

Trains the matrix-factorization backbone the Q-Former reads from, and reports
its own Valid/Test **uAUC** (a pure-CF reference point). The MF trainer derives
user/item counts from the data automatically.

In [ ]:
run(f'python -m sigllm.pipelines.rec.train_rec_baseline --cfg-path {CFG}')

## C. Q-Former Stage 1 — representation (ITC / ITM / ITG + item-item)

Pass the data-derived `user_num`/`item_num` as overrides so the frozen MF loads
with matching dimensions.

In [ ]:
run(f'python -m sigllm.pipelines.multimodal.train_qformer_stage1_representation --cfg-path {CFG} --options {OVR}')

## D. Q-Former Stage 2 — generative pretraining (LLM frozen)

In [ ]:
run(f'python -m sigllm.pipelines.multimodal.train_qformer_stage2_generative --cfg-path {CFG} --options {OVR}')

## E. Q-Former Stage 3 — CoLLM 2-step (LoRA, then Q-Former+proj CIE)

Step 1 trains LoRA on the text-only prompt; Step 2 trains the Q-Former + projection
on the multi-token CF prompt with LoRA frozen.

In [ ]:
run(f'python -m sigllm.pipelines.multimodal.train_qformer_stage3_step1_lora --cfg-path {CFG} --options {OVR}')

In [ ]:
run(f'python -m sigllm.pipelines.multimodal.train_qformer_stage3_step2_cie --cfg-path {CFG} --options {OVR}')

## F. Evaluation — uAUC overall + warm / cold

The Stage-3 runner evaluates `test`, `test_warm`, `test_cold` and logs **AUC/uAUC**
for each. Re-run Step 2 with `evaluate=True` and a trained `ckpt` to evaluate only.
Read the per-split `uAUC=...` lines from the log; cold uAUC is the headline for the
cold/sparse regime.

In [ ]:
# Evaluate-only on the trained Step-2 checkpoint.
STEP2_DIR = '/content/SigLLM/ckpt/qformer_stage3_step2_cie_amazon/'
CKPT = STEP2_DIR + 'checkpoint_best.pth'   # adjust if your runner names it differently
run(f'python -m sigllm.pipelines.multimodal.train_qformer_stage3_step2_cie --cfg-path {CFG} --options {OVR} run.evaluate=True model.ckpt={CKPT}')

## G. Ablations (all read out as uAUC)

These isolate the Q-Former's contribution — the core concern from the report.

| Arm | What it tests | How |
|---|---|---|
| **Title-free prompt** (2a) | Does the Q-Former carry item identity? | `prompt_path` -> `qformer_prompt_book_mt_notitle.txt` |
| **Ablate soft tokens** | Marginal value of CF tokens | `model.ablate_soft_tokens=True` |
| **cf_injection_mode=both** (2j) | Soft tokens + CoRA weight delta | `model.cf_injection_mode=both` |

If the title-free uAUC collapses toward 0.5 while the title prompt scores high, the
LLM was leaning on plain-text titles, not the Q-Former.

In [ ]:
NOTITLE = '/content/SigLLM/prompts/qformer_prompt_book_mt_notitle.txt'

# G.1 title-free control (CHANGE 2a)
run(f'python -m sigllm.pipelines.multimodal.train_qformer_stage3_step2_cie --cfg-path {CFG} --options {OVR} run.qformer_stage3_step2.prompt_path={NOTITLE} run.qformer_stage3_step2.output_dir=/content/SigLLM/ckpt/abl_notitle_amazon/')

In [ ]:
# G.2 ablate soft tokens at eval (load trained ckpt, zero soft tokens)
run(f'python -m sigllm.pipelines.multimodal.train_qformer_stage3_step2_cie --cfg-path {CFG} --options {OVR} run.evaluate=True model.ckpt={CKPT} model.ablate_soft_tokens=True')

In [ ]:
# G.3 cf_injection_mode=both (CHANGE 2j)
run(f'python -m sigllm.pipelines.multimodal.train_qformer_stage3_step2_cie --cfg-path {CFG} --options {OVR} model.cf_injection_mode=both run.qformer_stage3_step2.output_dir=/content/SigLLM/ckpt/abl_both_amazon/')

## H. Results (fill in)

| Model | Test uAUC | Warm uAUC | Cold uAUC |
|---|---|---|---|
| MF (CF only) | | | |
| SeLLa-Rec (baseline) | | | |
| Q-Former (title prompt) | | | |
| Q-Former (title-free, 2a) | | | |
| Q-Former − soft tokens (ablate) | | | |
| Q-Former (both injection, 2j) | | | |

_SeLLa-Rec baseline + SASRec substrate land in the next pass._